In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_using_large_models/10_3_transfer_learning.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Using Large Models

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2026
+ **Author:** Dr. Benedikt Zönnchen

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

In [ ]:
#@title Setup: install required Python packages

%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os

    folders = ['tests', 'data', 'models']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A3_existing_models"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('10_3_transfer_learning.ipynb')

## 28 Transfer Learning: Adapting a Pre-trained Model

### The core idea

So far we have trained models *from scratch*: all weights start at random values, and the model must learn everything from the training data alone. With a small dataset (~1000 melodies), this limits what the model can learn.

**Transfer learning** takes a different route: instead of starting from random weights, we start from a model that has *already* been trained — potentially on much more data, for much longer. We then **fine-tune** it on a new, smaller dataset.

The intuition: the pre-trained model has already learned general patterns (melody structure, note transitions, rhythm). Fine-tuning adapts these patterns towards a new musical style without discarding the learned foundation.

This is not just a technical trick. It mirrors how human musicians learn. A pianist who has studied classical music for years and then learns jazz does not start from zero: they transfer their knowledge of harmony, rhythm, and technique, and then adapt it to the new idiom.

### What gets updated during fine-tuning?

We can control which layers are updated:

- **Full fine-tuning**: update all parameters. Fast adaptation but may *catastrophically forget* what was learned before.
- **Partial fine-tuning (frozen body)**: freeze the early layers, only update the last few layers and the output head. The early layers retain general knowledge; only the style-specific layers change.
- **Feature extraction (frozen model)**: freeze everything, only train a new head on top. The pre-trained model is used as a fixed feature extractor.

In this notebook we compare these approaches on a concrete experiment: we fine-tune our folk-song transformer on **Bach chorales**, trying to nudge its style from German folk music towards Baroque counterpoint.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile, glob
import music21 as m21
from tqdm.auto import tqdm
from torch.utils.data import TensorDataset, DataLoader, random_split

from encoder import PianoRollEncoder, StringToIntEncoder, TERM_SYMBOL
from files import load_midi_files
from transformer import TransformerDecoder

In [ ]:
# Configure our plotting engine to get nice visualiziations
sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=0.8)
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

### 28.1 Loading the Pre-trained Folk-Song Model

We first recreate the vocabulary from the folk-song dataset and then load the weights saved in notebook 9_8.

In [ ]:
with zipfile.ZipFile('data/deu_folk_songs.zip', 'r') as z:
    z.extractall('data/deu_folk_songs/')

time_step = 0.5
mid_files = glob.glob('data/deu_folk_songs/**/*.mid', recursive=True)
folk_streams = load_midi_files(mid_files, time_step=time_step, transpose_to_major=True, max_files=1000)

piano_roll_encoder = PianoRollEncoder(time_step=time_step)
folk_rolls, _      = piano_roll_encoder.encode_streams(folk_streams)
string_to_int      = StringToIntEncoder(folk_rolls)
vocab_size         = len(string_to_int)
print(f'Vocabulary size: {vocab_size}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') # for mac gpu
sequence_len = 64
N_EMBD, N_HEADS, N_BLOCKS = 32, 4, 2

def make_model(dropout=0.2):
    return TransformerDecoder(
        vocab_size=vocab_size, sequence_len=sequence_len,
        n_embd=N_EMBD, n_heads=N_HEADS, n_blocks=N_BLOCKS,
        dropout=dropout
    ).to(device)

pretrained_model = make_model(dropout=0.0)
pretrained_model.load_state_dict(
    torch.load('models/transformer_model_1000_120.pt', map_location=device)
)
pretrained_model.eval()
print('Pre-trained folk-song model loaded.')

### 28.2 New Data: Bach Chorales

``music21`` includes a built-in corpus of J.S. Bach chorales — short four-voice harmonisations of Lutheran hymn melodies. We extract the **soprano voice** (the top part, carrying the melody) and encode it with the same piano-roll encoder we used for the folk songs.

This gives us a small but stylistically coherent dataset of Baroque melodies to fine-tune on.

In [ ]:
from music21 import corpus
from files import transpose   # re-use the key-normalisation helper

print('Loading Bach chorales from the music21 corpus…')
bach_works = corpus.getComposer('bach')

bach_streams = []
for path in bach_works:
    try:
        score = corpus.parse(path)
        if score.parts:
            soprano = score.parts[0]   # top voice
            soprano = transpose(soprano)
            bach_streams.append(soprano)
    except Exception:
        pass   # skip files that fail to parse
    if len(bach_streams) >= 60:
        break

print(f'Loaded {len(bach_streams)} Bach chorale soprano parts.')

In [ ]:
# Encode with the same encoder and vocabulary as the folk-song model
bach_rolls, skipped = piano_roll_encoder.encode_streams(bach_streams)
print(f'Encoded {len(bach_rolls)} Bach melodies ({len(skipped)} skipped due to time-step incompatibility).')

# Listen to the first Bach melody
print('First Bach melody (first 30 tokens):', bach_rolls[0][:30])
piano_roll_encoder.decode_stream(bach_rolls[0]).show('midi')

### 28.3 Preparing the Fine-tuning Dataset

The dataset preparation is identical to notebook 9_8: overlapping windows of length ``sequence_len``, with the target shifted by one position.

In [ ]:
def make_dataset(rolls, string_to_int, sequence_len, val_frac=0.2, batch_size=64):
    """Convert a list of piano rolls into train/val DataLoaders."""
    i_term = string_to_int.encode(TERM_SYMBOL)
    rolls_int = string_to_int.encode_sequences(rolls)

    xs, ys = [], []
    for roll in rolls_int:
        padded = [i_term] * sequence_len + roll + [i_term]
        for i in range(len(padded) - sequence_len):
            xs.append(padded[i : i + sequence_len])
            ys.append(padded[i + 1 : i + sequence_len + 1])

    X = torch.tensor(xs, dtype=torch.long)
    y = torch.tensor(ys, dtype=torch.long)
    dataset = TensorDataset(X, y)

    val_size   = int(len(dataset) * val_frac)
    train_size = len(dataset) - val_size
    gen        = torch.Generator().manual_seed(42)
    train_set, val_set = random_split(dataset, [train_size, val_size], generator=gen)

    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(val_set,   batch_size=batch_size))

train_loader, val_loader = make_dataset(bach_rolls, string_to_int, sequence_len)
print(f'Fine-tuning set: {len(train_loader.dataset)} train / {len(val_loader.dataset)} val windows')

### 28.4 Layer Freezing

In PyTorch, every parameter has a ``requires_grad`` flag. Setting it to ``False`` means the parameter will **not** be updated during the backward pass — it is *frozen*.

Our strategy:
- Freeze the token and position embedding tables (general note knowledge).
- Freeze the first transformer block (general sequence processing).
- Keep the last block and the output head (`lm_head`) trainable (style-specific adaptation).

---

🖍 **Exercise 28.1:** The cell below creates a copy of the pre-trained model for fine-tuning. Complete it by:
1. Setting ``requires_grad = False`` for **all** parameters.
2. Setting ``requires_grad = True`` for the **last transformer block** (``model.blocks[-1]``) and the **output head** (``model.lm_head``).

Store the number of trainable parameters in ``n_trainable``.

---

In [ ]:
import copy

# Create a fresh copy of the pre-trained model to fine-tune
model_ft = copy.deepcopy(pretrained_model)
model_ft.train()
model_ft = model_ft.to(device)

...

n_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model_ft.parameters())
print(f'Trainable parameters: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)')

In [ ]:
grader.check("q281")

### 28.5 Fine-tuning Loop

The training loop is the same as before — only the optimizer now sees the unfrozen parameters.

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 28.2:** Before running the fine-tuning loop, think about the following:

1. Why do we pass only ``filter(lambda p: p.requires_grad, model_ft.parameters())`` to the optimizer instead of all parameters?
2. We use a **smaller learning rate** (``1e-4``) than when training from scratch (``1e-3``). Why is a smaller learning rate recommended for fine-tuning?
3. We fine-tune for only 50 epochs on ~60 Bach melodies. How do you expect the loss curve to look compared to training from scratch on the same data?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->



In [ ]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_ft.parameters()),
    lr=1e-4
)

epochs = 50
train_losses, val_losses = [], []

def run_epoch(loader, model, train=True):
    model.train() if train else model.eval()
    total_loss, total = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits  = model(xb)
            B, T, C = logits.shape
            loss    = loss_fn(logits.view(B*T, C), yb.view(B*T))
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * B * T
            total      += B * T
    return total_loss / total

In [ ]:
for epoch in range(1, epochs + 1):
    tr = run_epoch(train_loader, model_ft, train=True)
    va = run_epoch(val_loader,   model_ft, train=False)
    train_losses.append(tr)
    val_losses.append(va)
    print(f'Epoch {epoch:2d}/{epochs}  train loss {tr:.4f}  val loss {va:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(x=range(len(train_losses)), y=train_losses, label='train', ax=ax)
sns.lineplot(x=range(len(val_losses)),   y=val_losses,   label='val',   ax=ax)
ax.set_title('Fine-tuning loss (frozen body + trainable last block & head)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
plt.tight_layout()
plt.show()

### 28.6 Before vs. After: Listening Comparison

Let us generate melodies with the *original* pre-trained model and the *fine-tuned* model and compare.

In [ ]:
def generate(model, seed, string_to_int, temperature=0.9, max_len=100):
    padded   = [TERM_SYMBOL] * sequence_len + seed
    seed_int = string_to_int.encode_sequence(padded)
    melody   = seed[:]
    model.eval()
    with torch.no_grad():
        while True:
            window = seed_int[-sequence_len:]
            idx    = torch.tensor([window], dtype=torch.long, device=device)
            logits = model(idx)[0, -1, :]
            probs  = torch.softmax(logits / temperature, dim=-1)
            sym_i  = torch.multinomial(probs, 1).item()
            seed_int.append(sym_i)
            sym = string_to_int.decode(sym_i)
            if sym == TERM_SYMBOL:
                break
            melody.append(sym)
            if len(melody) >= max_len:
                break
    return melody

SEED = []

In [ ]:
mel_folk = generate(pretrained_model, SEED, string_to_int)
print('Pre-trained (folk style):')
piano_roll_encoder.decode_stream(mel_folk).show('midi')

In [ ]:
mel_bach = generate(model_ft, SEED, string_to_int)
print('Fine-tuned on Bach chorales:')
piano_roll_encoder.decode_stream(mel_bach).show('midi')

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 28.3:** Compare the pre-trained and fine-tuned melodies.

1. Do you hear a difference in style between the two melodies? Describe it.
2. We fine-tuned on only the **last transformer block** while freezing the earlier layers. What might happen if you fine-tuned **all** layers instead? Would the result be better or worse? Try it by changing the code in Exercise 28.1 and re-running the training.
3. Transfer learning assumes that the pre-trained knowledge is useful for the new task. Can you imagine a scenario where the pre-trained weights would be *harmful* — where it would be better to train from scratch?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 28.7 Save the Fine-tuned Model (optional)

In [ ]:
import os
os.makedirs('models', exist_ok=True)
torch.save(model_ft.state_dict(), 'models/transformer_bach_ft.pt')
print('Fine-tuned model saved to models/transformer_bach_ft.pt')

### 28.8 Summary

| Approach | Dataset needed | Risk of forgetting | Training time |
|----------|---------------|-------------------|---------------|
| Train from scratch | Large | None | Long |
| Full fine-tuning | Small–medium | High (catastrophic forgetting) | Medium |
| Partial fine-tuning | Small | Low | Short |
| Feature extraction (head only) | Very small | None | Very short |

In the next notebook 

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_existing_models/10_4_lora.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a> 

we go one step further: instead of adapting the model to *one* new style, we teach it to *switch* between styles on demand.